# UK Public Procurement RAG System - BiP Solutions Tech Assignment

This notebook builds a Retrieval-Augmented Generation (RAG) system for UK Find a Tender Service procurement notices.  
The pipeline parses OCDS-style JSONL data, creates a clean contract-level dataset, retrieves relevant contracts using hybrid search, re-ranks results, generates grounded answers with a Hugging Face LLM, and evaluates retrieval quality using manual relevance labels.

## 1. Environment Setup

Firstly, we will install the Python libraries used by the retrieval, re-ranking, generation, and evaluation pipeline.

In [1]:
!pip install -q sentence-transformers chromadb rank-bm25 transformers accelerate bitsandbytes huggingface_hub

## 2. Data Ingestion and Flattening

The source file is a JSONL file where each line is one OCDS procurement notice/release.  
The functions below safely extract nested fields such as buyers, suppliers, CPV classifications, regions, values, statuses, and notice URLs into a flat table.

In [2]:
import json
import pandas as pd
from pathlib import Path


def safe_get(data, path, default=None):
    current = data

    for key in path:
        if isinstance(current, dict):
            current = current.get(key, default)
        elif isinstance(current, list) and isinstance(key, int):
            current = current[key] if len(current) > key else default
        else:
            return default

        if current is None:
            return default

    return current


def extract_suppliers(record):
    suppliers = []

    for award in record.get("awards", []):
        for supplier in award.get("suppliers", []):
            name = supplier.get("name")
            if name:
                suppliers.append(name)

    return sorted(set(suppliers))


def extract_cpv(record):
    cpv_codes = []
    cpv_descriptions = []

    classification = safe_get(record, ["tender", "classification"], {})
    if isinstance(classification, dict):
        if classification.get("id"):
            cpv_codes.append(classification["id"])
        if classification.get("description"):
            cpv_descriptions.append(classification["description"])

    for item in safe_get(record, ["tender", "items"], []):
        for cpv in item.get("additionalClassifications", []):
            if cpv.get("id"):
                cpv_codes.append(cpv["id"])
            if cpv.get("description"):
                cpv_descriptions.append(cpv["description"])

    return sorted(set(cpv_codes)), sorted(set(cpv_descriptions))


def extract_regions(record):
    regions = []

    for item in safe_get(record, ["tender", "items"], []):
        for address in item.get("deliveryAddresses", []):
            if address.get("region"):
                regions.append(address["region"])

        delivery_location = item.get("deliveryLocation", {})
        if isinstance(delivery_location, dict) and delivery_location.get("description"):
            regions.append(delivery_location["description"])

    return sorted(set(regions))


def extract_notice_url(record):
    documents = safe_get(record, ["tender", "documents"], [])

    for doc in documents:
        if doc.get("url"):
            return doc["url"]

    planning_docs = safe_get(record, ["planning", "documents"], [])

    for doc in planning_docs:
        if doc.get("url"):
            return doc["url"]

    return None


def extract_contract_values(record):
    values = []

    tender_amount = safe_get(record, ["tender", "value", "amount"])
    if tender_amount is not None:
        values.append(tender_amount)

    for contract in record.get("contracts", []):
        amount = safe_get(contract, ["value", "amount"])
        if amount is not None:
            values.append(amount)

    return values


def extract_award_statuses(record):
    return sorted(set([
        award.get("status")
        for award in record.get("awards", [])
        if award.get("status")
    ]))


def extract_contract_statuses(record):
    return sorted(set([
        contract.get("status")
        for contract in record.get("contracts", [])
        if contract.get("status")
    ]))


def flatten_ocds_record(record):
    cpv_codes, cpv_descriptions = extract_cpv(record)
    suppliers = extract_suppliers(record)
    regions = extract_regions(record)
    contract_values = extract_contract_values(record)

    title = safe_get(record, ["tender", "title"])
    description = safe_get(record, ["tender", "description"])
    buyer_name = safe_get(record, ["buyer", "name"])

    search_text = " ".join([
        str(title or ""),
        str(description or ""),
        str(buyer_name or ""),
        " ".join(suppliers),
        " ".join(cpv_descriptions),
        " ".join(regions),
        str(safe_get(record, ["tender", "procurementMethod"]) or ""),
        str(safe_get(record, ["tender", "mainProcurementCategory"]) or "")
    ])

    return {
        "ocid": record.get("ocid"),
        "notice_id": record.get("id"),
        "date": record.get("date"),

        "title": title,
        "description": description,

        "buyer_id": safe_get(record, ["buyer", "id"]),
        "buyer_name": buyer_name,

        "supplier_names": ", ".join(suppliers),

        "cpv_codes": ", ".join(cpv_codes),
        "cpv_descriptions": ", ".join(cpv_descriptions),

        "regions": ", ".join(regions),

        "tender_status": safe_get(record, ["tender", "status"]),
        "award_statuses": ", ".join(extract_award_statuses(record)),
        "contract_statuses": ", ".join(extract_contract_statuses(record)),

        "procurement_method": safe_get(record, ["tender", "procurementMethod"]),
        "procurement_method_details": safe_get(record, ["tender", "procurementMethodDetails"]),
        "main_procurement_category": safe_get(record, ["tender", "mainProcurementCategory"]),

        "contract_values": contract_values,
        "max_contract_value": max(contract_values) if contract_values else None,

        "notice_url": extract_notice_url(record),

        "search_text": search_text
    }


def read_jsonl_and_flatten(file_path):
    flattened_records = []
    failed_lines = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                flattened = flatten_ocds_record(record)
                flattened_records.append(flattened)

            except Exception as e:
                failed_lines.append({
                    "line_number": line_number,
                    "error": str(e)
                })

    df = pd.DataFrame(flattened_records)
    failed_df = pd.DataFrame(failed_lines)

    return df, failed_df

### 2.1 Read the JSONL File

This step converts the raw `united_kingdom_fts_2026.jsonl` file into a flat pandas DataFrame. Failed lines are captured separately for debugging.

In [3]:
file_path = "united_kingdom_fts_2026.jsonl"

df_contracts, failed_df = read_jsonl_and_flatten(file_path)

print("Total contracts processed:", len(df_contracts))
print("Failed lines:", len(failed_df))

df_contracts.head()

Total contracts processed: 35839
Failed lines: 0


,ocid,notice_id,date,title,description,buyer_id,buyer_name,supplier_names,cpv_codes,cpv_descriptions,...,tender_status,award_statuses,contract_statuses,procurement_method,procurement_method_details,main_procurement_category,contract_values,max_contract_value,notice_url,search_text
0,ocds-h6vhtk-02d3b4,ocds-h6vhtk-02d3b4-2026-01-07T10:39:21Z,2026-01-07T10:39:21Z,Adoption Support Fund Extension April - June 2026,This is a Corrigendum notice for the Adoption ...,GB-FTS-165,Department for Education,"Kinship, Mott Macdonald Limited",98000000,"Other community, social and personal services",...,complete,active,active,selective,Restricted procedure,services,"[1000000, 1865851, 3963174.42]",3.963174e+06,None,Adoption Support Fund Extension April - June 2...
1,ocds-h6vhtk-02e38e,ocds-h6vhtk-02e38e-2026-01-08T11:01:24Z,2026-01-08T11:01:24Z,Pseudo-DPS for Apprenticeship Training Provision,The Borough Council of Gateshead are an Appren...,GB-FTS-7372,GATESHEAD COUNCIL,,80000000,Education and training services,...,active,,,open,Open procedure,services,[3250000],3.250000e+06,None,Pseudo-DPS for Apprenticeship Training Provisi...
2,ocds-h6vhtk-034da5,ocds-h6vhtk-034da5-2026-01-05T10:41:31Z,2026-01-05T10:41:31Z,Provision of a Mixed Inpatient and Community R...,Provision of a mixed Inpatient and Community r...,GB-FTS-1287,Somerset NHS Foundation Trust,"Rethink Mental Illness, Rethink Mental Illness...",85323000,Community health services,...,complete,active,active,open,Open procedure,services,"[999344.0, 1500000.0, 2000000.0]",2.000000e+06,None,Provision of a Mixed Inpatient and Community R...
3,ocds-h6vhtk-035235,ocds-h6vhtk-035235-2026-01-05T16:39:00Z,2026-01-05T16:39:00Z,"The Provision of Marketing, Campaigns, and Med...",The Provision of UG Student Recruitment Market...,GB-COH-06023243,UNIVERSITY OF GLOUCESTERSHIRE,"Hybrid News Limited, Hybrid News Ltd",79340000,Advertising and marketing services,...,complete,active,active,selective,Restricted procedure,services,[2580000],2.580000e+06,None,"The Provision of Marketing, Campaigns, and Med..."
4,ocds-h6vhtk-0354e4,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,2026-01-06T14:46:47Z,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions on behalf of Surrey a...,GB-CHC-1126477,NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...","32332100, 79500000","Dictating machines, Office-support services",...,complete,active,active,selective,Restricted procedure,services,"[200000000, 75000000.0, 65000000.0, 35000000.0...",2.000000e+08,None,"Extension- Digital Dictation, Speech/Voice Rec..."


## 3. Data Cleaning and Normalisation

This step converts empty strings into real missing values, standardises text fields, parses dates, converts contract values to numeric format, and adds useful metadata flags.

In [4]:
import numpy as np
import pandas as pd

# 1. Replace empty strings with NaN
df_contracts = df_contracts.replace("", np.nan)

# 2. Clean whitespace in text columns
text_columns = [
    "title",
    "description",
    "buyer_name",
    "supplier_names",
    "cpv_codes",
    "cpv_descriptions",
    "regions",
    "tender_status",
    "award_statuses",
    "contract_statuses",
    "procurement_method",
    "procurement_method_details",
    "main_procurement_category",
    "notice_url",
    "search_text"
]

for col in text_columns:
    if col in df_contracts.columns:
        df_contracts[col] = (
            df_contracts[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

# 3. Convert date column
df_contracts["date"] = pd.to_datetime(df_contracts["date"], errors="coerce", utc=True)

# 4. Convert contract value column
df_contracts["max_contract_value"] = pd.to_numeric(
    df_contracts["max_contract_value"],
    errors="coerce"
)

# 5. Add useful boolean flags
df_contracts["has_supplier"] = df_contracts["supplier_names"].notna()
df_contracts["has_cpv"] = df_contracts["cpv_codes"].notna()
df_contracts["has_region"] = df_contracts["regions"].notna()
df_contracts["has_contract_value"] = df_contracts["max_contract_value"].notna()

# 6. Create contract year
df_contracts["year"] = df_contracts["date"].dt.year

print(df_contracts.shape)
df_contracts.head()

(35839, 26)


,ocid,notice_id,date,title,description,buyer_id,buyer_name,supplier_names,cpv_codes,cpv_descriptions,...,main_procurement_category,contract_values,max_contract_value,notice_url,search_text,has_supplier,has_cpv,has_region,has_contract_value,year
0,ocds-h6vhtk-02d3b4,ocds-h6vhtk-02d3b4-2026-01-07T10:39:21Z,2026-01-07 10:39:21+00:00,Adoption Support Fund Extension April - June 2026,This is a Corrigendum notice for the Adoption ...,GB-FTS-165,Department for Education,"Kinship, Mott Macdonald Limited",98000000,"Other community, social and personal services",...,services,"[1000000, 1865851, 3963174.42]",3.963174e+06,<NA>,Adoption Support Fund Extension April - June 2...,True,True,True,True,2026
1,ocds-h6vhtk-02e38e,ocds-h6vhtk-02e38e-2026-01-08T11:01:24Z,2026-01-08 11:01:24+00:00,Pseudo-DPS for Apprenticeship Training Provision,The Borough Council of Gateshead are an Appren...,GB-FTS-7372,GATESHEAD COUNCIL,<NA>,80000000,Education and training services,...,services,[3250000],3.250000e+06,<NA>,Pseudo-DPS for Apprenticeship Training Provisi...,False,True,True,True,2026
2,ocds-h6vhtk-034da5,ocds-h6vhtk-034da5-2026-01-05T10:41:31Z,2026-01-05 10:41:31+00:00,Provision of a Mixed Inpatient and Community R...,Provision of a mixed Inpatient and Community r...,GB-FTS-1287,Somerset NHS Foundation Trust,"Rethink Mental Illness, Rethink Mental Illness...",85323000,Community health services,...,services,"[999344.0, 1500000.0, 2000000.0]",2.000000e+06,<NA>,Provision of a Mixed Inpatient and Community R...,True,True,True,True,2026
3,ocds-h6vhtk-035235,ocds-h6vhtk-035235-2026-01-05T16:39:00Z,2026-01-05 16:39:00+00:00,"The Provision of Marketing, Campaigns, and Med...",The Provision of UG Student Recruitment Market...,GB-COH-06023243,UNIVERSITY OF GLOUCESTERSHIRE,"Hybrid News Limited, Hybrid News Ltd",79340000,Advertising and marketing services,...,services,[2580000],2.580000e+06,<NA>,"The Provision of Marketing, Campaigns, and Med...",True,True,True,True,2026
4,ocds-h6vhtk-0354e4,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,2026-01-06 14:46:47+00:00,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions on behalf of Surrey a...,GB-CHC-1126477,NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...","32332100, 79500000","Dictating machines, Office-support services",...,services,"[200000000, 75000000.0, 65000000.0, 35000000.0...",2.000000e+08,<NA>,"Extension- Digital Dictation, Speech/Voice Rec...",True,True,True,True,2026


### 3.1 Build Structured Search Text

The `search_text` field is built using labelled field context. This improves embedding quality because the model can distinguish title, buyer, supplier, CPV category, region, and procurement method.

In [5]:
def build_structured_search_text(row):
    parts = []

    if pd.notna(row.get("title")):
        parts.append(f"Title: {row['title']}")

    if pd.notna(row.get("description")):
        parts.append(f"Description: {row['description']}")

    if pd.notna(row.get("buyer_name")):
        parts.append(f"Buyer: {row['buyer_name']}")

    if pd.notna(row.get("supplier_names")):
        parts.append(f"Suppliers: {row['supplier_names']}")

    if pd.notna(row.get("cpv_descriptions")):
        parts.append(f"CPV category: {row['cpv_descriptions']}")

    if pd.notna(row.get("cpv_codes")):
        parts.append(f"CPV code: {row['cpv_codes']}")

    if pd.notna(row.get("regions")):
        parts.append(f"Region: {row['regions']}")

    if pd.notna(row.get("procurement_method")):
        parts.append(f"Procurement method: {row['procurement_method']}")

    if pd.notna(row.get("procurement_method_details")):
        parts.append(f"Procurement method details: {row['procurement_method_details']}")

    if pd.notna(row.get("main_procurement_category")):
        parts.append(f"Main procurement category: {row['main_procurement_category']}")

    if pd.notna(row.get("max_contract_value")):
        parts.append(f"Maximum contract value GBP: {row['max_contract_value']}")

    return "\n".join(parts)


df_contracts["search_text"] = df_contracts.apply(build_structured_search_text, axis=1)

### 3.2 Sanity Checks

These checks confirm row counts, uniqueness, and metadata coverage before moving into retrieval.

In [6]:
print("Total rows:", len(df_contracts))
print("Unique OCIDs:", df_contracts["ocid"].nunique())
print("Duplicate OCIDs:", df_contracts["ocid"].duplicated().sum())
print("Unique Notice IDs:", df_contracts["notice_id"].nunique())


Total rows: 35839
Unique OCIDs: 35839
Duplicate OCIDs: 0
Unique Notice IDs: 35839


### 3.3 Save Final Retrieval Dataset

The cleaned table is saved as `final_retrieval_contracts_2026.csv`. This becomes the input for the RAG pipeline.

In [7]:
# one row per notice/contract record.
# The dataset already has unique OCIDs and notice IDs, so no deduplication is required.

df_retrieval = df_contracts.copy()

# Droping list-style column because it is not required for retrieval.
if "contract_values" in df_retrieval.columns:
    df_retrieval = df_retrieval.drop(columns=["contract_values"])

df_retrieval.to_csv("final_retrieval_contracts_2026.csv", index=False)

print(df_retrieval.shape)
df_retrieval.head()

(35839, 25)


,ocid,notice_id,date,title,description,buyer_id,buyer_name,supplier_names,cpv_codes,cpv_descriptions,...,procurement_method_details,main_procurement_category,max_contract_value,notice_url,search_text,has_supplier,has_cpv,has_region,has_contract_value,year
0,ocds-h6vhtk-02d3b4,ocds-h6vhtk-02d3b4-2026-01-07T10:39:21Z,2026-01-07 10:39:21+00:00,Adoption Support Fund Extension April - June 2026,This is a Corrigendum notice for the Adoption ...,GB-FTS-165,Department for Education,"Kinship, Mott Macdonald Limited",98000000,"Other community, social and personal services",...,Restricted procedure,services,3.963174e+06,<NA>,Title: Adoption Support Fund Extension April -...,True,True,True,True,2026
1,ocds-h6vhtk-02e38e,ocds-h6vhtk-02e38e-2026-01-08T11:01:24Z,2026-01-08 11:01:24+00:00,Pseudo-DPS for Apprenticeship Training Provision,The Borough Council of Gateshead are an Appren...,GB-FTS-7372,GATESHEAD COUNCIL,<NA>,80000000,Education and training services,...,Open procedure,services,3.250000e+06,<NA>,Title: Pseudo-DPS for Apprenticeship Training ...,False,True,True,True,2026
2,ocds-h6vhtk-034da5,ocds-h6vhtk-034da5-2026-01-05T10:41:31Z,2026-01-05 10:41:31+00:00,Provision of a Mixed Inpatient and Community R...,Provision of a mixed Inpatient and Community r...,GB-FTS-1287,Somerset NHS Foundation Trust,"Rethink Mental Illness, Rethink Mental Illness...",85323000,Community health services,...,Open procedure,services,2.000000e+06,<NA>,Title: Provision of a Mixed Inpatient and Comm...,True,True,True,True,2026
3,ocds-h6vhtk-035235,ocds-h6vhtk-035235-2026-01-05T16:39:00Z,2026-01-05 16:39:00+00:00,"The Provision of Marketing, Campaigns, and Med...",The Provision of UG Student Recruitment Market...,GB-COH-06023243,UNIVERSITY OF GLOUCESTERSHIRE,"Hybrid News Limited, Hybrid News Ltd",79340000,Advertising and marketing services,...,Restricted procedure,services,2.580000e+06,<NA>,"Title: The Provision of Marketing, Campaigns, ...",True,True,True,True,2026
4,ocds-h6vhtk-0354e4,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,2026-01-06 14:46:47+00:00,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions on behalf of Surrey a...,GB-CHC-1126477,NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...","32332100, 79500000","Dictating machines, Office-support services",...,Restricted procedure,services,2.000000e+08,<NA>,"Title: Extension- Digital Dictation, Speech/Vo...",True,True,True,True,2026


## 4. Load Retrieval Dataset

From this point onward, the notebook uses the cleaned contract-level table rather than the raw JSONL file.

In [8]:
df = pd.read_csv("final_retrieval_contracts_2026.csv")
df = df.replace({np.nan: None})

print(df.shape)
df.head()

(35839, 25)


,ocid,notice_id,date,title,description,buyer_id,buyer_name,supplier_names,cpv_codes,cpv_descriptions,...,procurement_method_details,main_procurement_category,max_contract_value,notice_url,search_text,has_supplier,has_cpv,has_region,has_contract_value,year
0,ocds-h6vhtk-02d3b4,ocds-h6vhtk-02d3b4-2026-01-07T10:39:21Z,2026-01-07 10:39:21+00:00,Adoption Support Fund Extension April - June 2026,This is a Corrigendum notice for the Adoption ...,GB-FTS-165,Department for Education,"Kinship, Mott Macdonald Limited",98000000,"Other community, social and personal services",...,Restricted procedure,services,3963174.42,None,Title: Adoption Support Fund Extension April -...,True,True,True,True,2026
1,ocds-h6vhtk-02e38e,ocds-h6vhtk-02e38e-2026-01-08T11:01:24Z,2026-01-08 11:01:24+00:00,Pseudo-DPS for Apprenticeship Training Provision,The Borough Council of Gateshead are an Appren...,GB-FTS-7372,GATESHEAD COUNCIL,None,80000000,Education and training services,...,Open procedure,services,3250000.0,None,Title: Pseudo-DPS for Apprenticeship Training ...,False,True,True,True,2026
2,ocds-h6vhtk-034da5,ocds-h6vhtk-034da5-2026-01-05T10:41:31Z,2026-01-05 10:41:31+00:00,Provision of a Mixed Inpatient and Community R...,Provision of a mixed Inpatient and Community r...,GB-FTS-1287,Somerset NHS Foundation Trust,"Rethink Mental Illness, Rethink Mental Illness...",85323000,Community health services,...,Open procedure,services,2000000.0,None,Title: Provision of a Mixed Inpatient and Comm...,True,True,True,True,2026
3,ocds-h6vhtk-035235,ocds-h6vhtk-035235-2026-01-05T16:39:00Z,2026-01-05 16:39:00+00:00,"The Provision of Marketing, Campaigns, and Med...",The Provision of UG Student Recruitment Market...,GB-COH-06023243,UNIVERSITY OF GLOUCESTERSHIRE,"Hybrid News Limited, Hybrid News Ltd",79340000,Advertising and marketing services,...,Restricted procedure,services,2580000.0,None,"Title: The Provision of Marketing, Campaigns, ...",True,True,True,True,2026
4,ocds-h6vhtk-0354e4,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,2026-01-06 14:46:47+00:00,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions on behalf of Surrey a...,GB-CHC-1126477,NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...","32332100, 79500000","Dictating machines, Office-support services",...,Restricted procedure,services,200000000.0,None,"Title: Extension- Digital Dictation, Speech/Vo...",True,True,True,True,2026


### 4.1 Build RAG Text

`rag_text` is the main text representation used for embeddings, BM25, and LLM context. It contains the most useful searchable fields from each contract.

In [9]:
def safe_text(value):
    return "" if value is None or pd.isna(value) else str(value)

def build_rag_text(row):
    return f"""
Title: {safe_text(row.get('title'))}
Description: {safe_text(row.get('description'))}
Buyer: {safe_text(row.get('buyer_name'))}
Suppliers: {safe_text(row.get('supplier_names'))}
CPV Codes: {safe_text(row.get('cpv_codes'))}
CPV Categories: {safe_text(row.get('cpv_descriptions'))}
Regions: {safe_text(row.get('regions'))}
Procurement Method: {safe_text(row.get('procurement_method'))}
Procurement Details: {safe_text(row.get('procurement_method_details'))}
Main Category: {safe_text(row.get('main_procurement_category'))}
Value GBP: {safe_text(row.get('max_contract_value'))}
""".strip()

df["rag_text"] = df.apply(build_rag_text, axis=1)

## 5. Query Intent Parsing

The query parser extracts simple structured signals from the user's natural language query, such as NHS buyer context, region terms, CPV codes, category terms, framework requirements, and multi-supplier requirements.

In [10]:
def parse_query_intent(query):
    q = query.lower()

    intent = {
        "raw_query": query,
        "buyer_contains": None,
        "supplier_contains": None,
        "category_terms": [],
        "region_terms": [],
        "cpv_code": None,
        "framework_required": False,
        "multi_supplier_required": False
    }

    if "nhs" in q:
        intent["buyer_contains"] = "nhs"

    if "london" in q:
        intent["region_terms"].extend(["london", "uki"])

    if "scotland" in q:
        intent["region_terms"].extend(["scotland", "ukm"])

    if "wales" in q:
        intent["region_terms"].extend(["wales", "ukl"])

    if "construction" in q:
        intent["category_terms"].extend(["construction", "works", "building"])

    if "it" in q or "technology" in q or "software" in q:
        intent["category_terms"].extend(["it", "technology", "software", "computer", "digital"])

    if "cloud" in q:
        intent["category_terms"].extend(["cloud", "hosting", "saas", "software"])

    if "framework" in q:
        intent["framework_required"] = True

    if "multi supplier" in q or "multi-supplier" in q:
        intent["multi_supplier_required"] = True

    cpv_match = re.search(r"\b\d{8}\b", q)
    if cpv_match:
        intent["cpv_code"] = cpv_match.group(0)

    supplier_match = re.search(r"awarded to ([a-zA-Z0-9 &.-]+)", q)
    if supplier_match:
        intent["supplier_contains"] = supplier_match.group(1).strip()

    return intent

### 5.1 Metadata / Intent Pre-filtering

Before ranking, the system applies lightweight filters based on the parsed query intent. If filters become too strict, it falls back to the full dataset.

In [11]:
def apply_intent_prefilter(df, intent):
    filtered = df.copy()

    combined_text = (
        filtered["rag_text"]
        .fillna("")
        .str.lower()
    )

    if intent["buyer_contains"]:
        filtered = filtered[
            filtered["buyer_name"]
            .fillna("")
            .str.lower()
            .str.contains(intent["buyer_contains"], na=False)
        ]

    if intent["supplier_contains"]:
        filtered = filtered[
            filtered["supplier_names"]
            .fillna("")
            .str.lower()
            .str.contains(intent["supplier_contains"], na=False)
        ]

    if intent["cpv_code"]:
        filtered = filtered[
            filtered["cpv_codes"]
            .fillna("")
            .str.contains(intent["cpv_code"], na=False)
        ]

    if intent["category_terms"]:
        pattern = "|".join(intent["category_terms"])
        filtered = filtered[
            filtered["rag_text"]
            .fillna("")
            .str.lower()
            .str.contains(pattern, na=False)
        ]

    if intent["region_terms"]:
        pattern = "|".join(intent["region_terms"])
        filtered = filtered[
            filtered["rag_text"]
            .fillna("")
            .str.lower()
            .str.contains(pattern, na=False)
        ]

    if intent["framework_required"]:
        filtered = filtered[
            filtered["rag_text"]
            .fillna("")
            .str.lower()
            .str.contains("framework", na=False)
        ]

    if intent["multi_supplier_required"]:
        filtered = filtered[
            filtered["supplier_names"]
            .fillna("")
            .str.contains(",", na=False)
        ]

    # Fallback: if filters are too strict, return full dataset
    if len(filtered) < 5:
        return df.copy()

    return filtered

## 6. Vector Store with Hugging Face Embeddings and ChromaDB

The system uses `BAAI/bge-small-en-v1.5`, a Hugging Face embedding model, to represent procurement notices as dense vectors. ChromaDB stores embeddings and metadata for vector retrieval.

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_fts_db")

collection = client.get_or_create_collection(
    name="fts_contracts"
)

print("Existing Chroma records:", collection.count())

Existing Chroma records: 35839


In [14]:
documents = df["rag_text"].tolist()
ids = df["notice_id"].astype(str).tolist()

metadatas = []

for _, row in df.iterrows():
    metadatas.append({
        "notice_id": safe_text(row.get("notice_id")),
        "title": safe_text(row.get("title")),
        "buyer_name": safe_text(row.get("buyer_name")),
        "supplier_names": safe_text(row.get("supplier_names")),
        "cpv_codes": safe_text(row.get("cpv_codes")),
        "cpv_descriptions": safe_text(row.get("cpv_descriptions")),
        "regions": safe_text(row.get("regions")),
        "procurement_method": safe_text(row.get("procurement_method")),
        "max_contract_value": safe_text(row.get("max_contract_value"))
    })

batch_size = 500

# Only populate Chroma if the collection is empty.
# This prevents duplicate ID errors when the notebook is rerun.
if collection.count() == 0:
    for start in range(0, len(documents), batch_size):
        end = start + batch_size

        batch_docs = documents[start:end]
        batch_ids = ids[start:end]
        batch_meta = metadatas[start:end]

        batch_embeddings = embedding_model.encode(
            batch_docs,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

        collection.add(
            documents=batch_docs,
            embeddings=batch_embeddings,
            metadatas=batch_meta,
            ids=batch_ids
        )

print("Chroma count:", collection.count())

Chroma count: 35839


## 7. BM25 Keyword Retrieval

BM25 complements vector search by handling exact keywords, supplier names, CPV codes, and short precise queries.

In [15]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b[a-zA-Z0-9]+\b", str(text).lower())

tokenized_corpus = [tokenize(text) for text in df["rag_text"].tolist()]

bm25 = BM25Okapi(tokenized_corpus)

## 8. Hybrid Retrieval with Reciprocal Rank Fusion

The system combines vector results and BM25 results using Reciprocal Rank Fusion (RRF). This gives a more robust candidate set than either retrieval method alone.

In [16]:
def reciprocal_rank_fusion(rank_lists, k=60):
    scores = {}

    for rank_list in rank_lists:
        for rank, doc_id in enumerate(rank_list):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [17]:
def hybrid_retrieve(query, top_k=5, candidate_k=50):
    intent = parse_query_intent(query)

    filtered_df = apply_intent_prefilter(df, intent)

    allowed_ids = set(filtered_df["notice_id"].astype(str).tolist())

    # Vector search
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()[0]

    vector_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=candidate_k
    )

    vector_ids = vector_results["ids"][0]
    vector_distances = vector_results["distances"][0]

    vector_ranked_ids = [
        doc_id for doc_id in vector_ids
        if doc_id in allowed_ids
    ]

    # BM25 search
    query_tokens = tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)

    bm25_ranked_indices = np.argsort(bm25_scores)[::-1][:candidate_k * 3]

    bm25_ranked_ids = [
        str(df.iloc[i]["notice_id"])
        for i in bm25_ranked_indices
        if str(df.iloc[i]["notice_id"]) in allowed_ids
    ][:candidate_k]

    # RRF fusion
    fused = reciprocal_rank_fusion(
        [vector_ranked_ids, bm25_ranked_ids]
    )

    final_ids = [doc_id for doc_id, score in fused[:top_k]]
    fusion_scores = dict(fused)

    results = df[
        df["notice_id"].astype(str).isin(final_ids)
    ].copy()

    results["rrf_score"] = results["notice_id"].astype(str).map(fusion_scores)

    results = results.sort_values("rrf_score", ascending=False)

    return results, intent, vector_ranked_ids, bm25_ranked_ids

## 9. Result Summaries and Match Explanations

These helper functions create extractive summaries and explain why each contract matched the query using semantic retrieval, BM25, keyword overlap, metadata filters, and CPV context.

In [18]:
def extractive_summary(text, max_words=70):
    words = str(text).split()
    return " ".join(words[:max_words]) + ("..." if len(words) > max_words else "")

def keyword_overlap(query, text):
    query_terms = set(tokenize(query))
    text_terms = set(tokenize(text))

    stopwords = {
        "the", "and", "for", "with", "from", "find", "show",
        "contract", "contracts", "service", "services", "awarded"
    }

    query_terms = query_terms - stopwords

    matched = sorted(query_terms.intersection(text_terms))

    return matched

def explain_match(row, query, intent):
    matched_terms = keyword_overlap(query, row["rag_text"])

    explanations = []

    explanations.append("Retrieved using hybrid semantic vector search and BM25 keyword search.")

    if matched_terms:
        explanations.append(
            "Keyword overlap: " + ", ".join(matched_terms[:10])
        )

    if intent["buyer_contains"]:
        explanations.append(
            f"Buyer filter/context matched: {intent['buyer_contains']}"
        )

    if intent["category_terms"]:
        explanations.append(
            "Category intent terms considered: " + ", ".join(intent["category_terms"])
        )

    if intent["region_terms"]:
        explanations.append(
            "Region intent terms considered: " + ", ".join(intent["region_terms"])
        )

    if intent["framework_required"]:
        explanations.append("Framework requirement identified in query.")

    if intent["multi_supplier_required"]:
        explanations.append("Multi-supplier requirement identified in query.")

    if row.get("cpv_descriptions"):
        explanations.append(
            f"CPV context: {row.get('cpv_descriptions')}"
        )

    return " | ".join(explanations)

## 10. Cross-Encoder Re-ranking

A Hugging Face cross-encoder re-ranks the hybrid candidate set by scoring the query and each candidate contract together. This improves top-result quality compared with vector search alone.

In [19]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [20]:
def rerank_results(query, results, top_k=5):
    pairs = []

    for _, row in results.iterrows():
        pairs.append([
            query,
            row["rag_text"]
        ])

    rerank_scores = reranker.predict(pairs)

    results = results.copy()
    results["rerank_score"] = rerank_scores

    results = results.sort_values(
        "rerank_score",
        ascending=False
    )

    return results.head(top_k)

In [21]:
def hybrid_retrieve_with_reranking(query, top_k=5, candidate_k=100):
    results, intent, vector_ids, bm25_ids = hybrid_retrieve(
        query=query,
        top_k=candidate_k,
        candidate_k=candidate_k
    )

    reranked_results = rerank_results(
        query=query,
        results=results,
        top_k=top_k
    )

    return reranked_results, intent

In [22]:
def explain_match_v2(row, query, intent):
    matched_terms = keyword_overlap(query, row["rag_text"])

    explanation_parts = []

    explanation_parts.append(
        f"Semantic + BM25 candidate retrieval followed by cross-encoder re-ranking."
    )

    explanation_parts.append(
        f"Cross-encoder relevance score: {row['rerank_score']:.3f}"
    )

    if row.get("rrf_score") is not None:
        explanation_parts.append(
            f"Hybrid RRF score: {row['rrf_score']:.4f}"
        )

    if matched_terms:
        explanation_parts.append(
            "Keyword overlap: " + ", ".join(matched_terms[:10])
        )

    if intent.get("buyer_contains"):
        explanation_parts.append(
            f"Buyer intent matched/considered: {intent['buyer_contains']}"
        )

    if intent.get("category_terms"):
        explanation_parts.append(
            "Category intent considered: " + ", ".join(intent["category_terms"])
        )

    if intent.get("region_terms"):
        explanation_parts.append(
            "Region intent considered: " + ", ".join(intent["region_terms"])
        )

    if intent.get("framework_required"):
        explanation_parts.append(
            "Query requested framework-related contracts."
        )

    if intent.get("multi_supplier_required"):
        explanation_parts.append(
            "Query requested multi-supplier contracts/frameworks."
        )

    if pd.notna(row.get("cpv_descriptions")):
        explanation_parts.append(
            f"CPV context: {row['cpv_descriptions']}"
        )

    return " | ".join(explanation_parts)

In [23]:
def rag_search_v2(query, top_k=5):
    results, intent = hybrid_retrieve_with_reranking(
        query=query,
        top_k=top_k,
        candidate_k=150
    )

    output = []

    for _, row in results.iterrows():
        output.append({
            "Notice ID": row.get("notice_id"),
            "Title": row.get("title"),
            "Buyer": row.get("buyer_name"),
            "Suppliers": row.get("supplier_names"),
            "Summary": extractive_summary(row.get("description"), max_words=80),
            "Relevance Score": round(float(row.get("rerank_score", 0)), 4),
            "Hybrid Score": round(float(row.get("rrf_score", 0)), 4),
            "Why it matched": explain_match_v2(row, query, intent)
        })

    return pd.DataFrame(output)

### 10.1 Test Retrieval on Assessment Queries

These are the three sample queries from the technical assessment brief.

In [24]:
sample_queries = [
    "Find framework agreements for IT services in the NHS",
    "Contracts awarded in London for construction",
    "Multi-supplier frameworks for cloud services"
]

for q in sample_queries:
    print("\n" + "=" * 100)
    print("QUERY:", q)
    display(rag_search_v2(q, top_k=5))


QUERY: Find framework agreements for IT services in the NHS


,Notice ID,Title,Buyer,Suppliers,Summary,Relevance Score,Hybrid Score,Why it matched
0,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...",NHS Commercial Solutions on behalf of Surrey a...,5.8426,0.0156,Semantic + BM25 candidate retrieval followed b...
1,ocds-h6vhtk-064cfe-2026-02-10T15:54:12Z,Consultancy and Advisory Services,NHS Shared Business Services Limited,None,This proposed Framework Agreement will aim to ...,5.7832,0.0164,Semantic + BM25 candidate retrieval followed b...
2,ocds-h6vhtk-02ba8e-2026-03-30T12:12:07+01:00,Digital Document Solutions (DDS) Framework Agr...,Guy's and St Thomas' NHS Foundation Trust,"3M MModal, ASL, Academia, Altodigital Networks...",The Digital Document Solutions (DDS) Framework...,5.6225,0.0230,Semantic + BM25 candidate retrieval followed b...
3,ocds-h6vhtk-058e51-2026-03-26T13:16:58Z,National Framework Agreement for the provision...,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,"Amvale Medical Transport Ltd, LSA Secure Ltd t...",The Countess of Chester Hospital NHS Foundatio...,5.3214,0.0258,Semantic + BM25 candidate retrieval followed b...
4,ocds-h6vhtk-0694e3-2026-05-08T15:04:53+01:00,Framework Agreement for the Provision of Leade...,NHS Wales Shared Services Partnership-Procurem...,None,Aneurin Bevan University Health Board (ABUHB) ...,5.2313,0.0251,Semantic + BM25 candidate retrieval followed b...



QUERY: Contracts awarded in London for construction


,Notice ID,Title,Buyer,Suppliers,Summary,Relevance Score,Hybrid Score,Why it matched
0,ocds-h6vhtk-066354-2026-03-12T12:14:29Z,Measured Term Contract 2026,The Mayor and Commonalty and Citizens of the C...,None,The City of London Corporation intends to esta...,4.8668,0.0164,Semantic + BM25 candidate retrieval followed b...
1,ocds-h6vhtk-06085c-2026-01-19T10:05:18Z,Carpenters Road – Road Safety Audit and Design...,London Legacy Development Corporation,Urban Movement LIMITED,The London Legacy Development Corporation (LLD...,4.4562,0.0113,Semantic + BM25 candidate retrieval followed b...
2,ocds-h6vhtk-065731-2026-02-18T14:38:00Z,"Building Remediation Project, London SW4",Metropolitan Housing Trust Limited,Architectural Decorators Limited,Procurement of a Principal Contractor to under...,4.2011,0.0159,Semantic + BM25 candidate retrieval followed b...
3,ocds-h6vhtk-06578d-2026-02-18T17:26:52Z,Building Remediation Project,Metropolitan Housing Trust Limited,Higgins Partnerships 1961 PLC,Procurement of a Principal Contractor to under...,4.1504,0.0078,Semantic + BM25 candidate retrieval followed b...
4,ocds-h6vhtk-04e6d8-2026-03-19T09:28:46Z,London Construction Programme General Works Fr...,London Borough of Haringey,"Advanced Interior Solutions Ltd., Alcema Ltd, ...",The London Construction Programme (LCP) on beh...,4.1153,0.0241,Semantic + BM25 candidate retrieval followed b...



QUERY: Multi-supplier frameworks for cloud services


,Notice ID,Title,Buyer,Suppliers,Summary,Relevance Score,Hybrid Score,Why it matched
0,ocds-h6vhtk-05ee31-2026-03-26T10:37:40Z,BLC0306 - Framework for the Provision of Cover...,BlueLight Commercial Limited,"ABM Intelligence Ltd, CTRL O LTD, NEC Software...",BlueLight Commercial has established a nationa...,2.8861,0.0164,Semantic + BM25 candidate retrieval followed b...
1,ocds-h6vhtk-0553a6-2026-05-07T10:37:36+01:00,BLC-0201 - Cyber Security Services Framework,BLUELIGHT COMMERCIAL LIMITED,"ACTICA CONSULTING LIMITED, AMENTUM CLEAN ENERG...",BlueLight Commercial are establishing a multi-...,2.3677,0.0152,Semantic + BM25 candidate retrieval followed b...
2,ocds-h6vhtk-050458-2026-02-27T10:40:20Z,512_25 Utility Metering and Data Services,"Leicestershire County Council, trading as ESPO","NPOWER COMMERCIAL GAS LIMITED, SMS ENERGY SERV...",A framework agreement covering Half Hourly Ele...,2.0724,0.0156,Semantic + BM25 candidate retrieval followed b...
3,ocds-h6vhtk-05f100-2026-05-01T11:35:34+01:00,NHS LPP Asset Utilisation Framework Agreement,Guy's and St Thomas' NHS Foundation Trust,"Add Latent Ltd, Asset Workflows Limited, INNAX...",A framework opportunity for experienced and qu...,1.7821,0.0161,Semantic + BM25 candidate retrieval followed b...
4,ocds-h6vhtk-059d98-2026-02-16T09:34:56Z,Software Reseller Framework,YORKSHIRE WATER SERVICES LIMITED,"Italik Limited, SCC (UK) LIMITED, SOFTCAT PLC,...",Yorkshire Water Services (YWS) is looking for ...,1.0444,0.0313,Semantic + BM25 candidate retrieval followed b...


## 11. Grounded Answer Generation with Mistral

The retrieved contracts are passed to `mistralai/Mistral-7B-Instruct-v0.3` to generate a grounded natural language answer.  
The prompt instructs the model to use only retrieved evidence and to mention limitations when results are partial matches.

In [25]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub

In [26]:
from huggingface_hub import login

# Paste your Hugging Face read token when prompted.
# Do not hard-code API tokens in the notebook before pushing to GitHub.
login()


In [27]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
Tesla T4


In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

mistral_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

mistral_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [29]:
def build_mistral_context(results):
    context_blocks = []

    for i, (_, row) in enumerate(results.iterrows(), start=1):
        block = f"""
Contract {i}
Notice ID: {row.get("Notice ID")}
Title: {row.get("Title")}
Buyer: {row.get("Buyer")}
Suppliers: {row.get("Suppliers")}
Summary: {row.get("Summary")}
Relevance Score: {row.get("Relevance Score")}
Hybrid Score: {row.get("Hybrid Score")}
Why it matched: {row.get("Why it matched")}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

In [30]:
def generate_mistral_answer(query, results):
    context = build_mistral_context(results)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a UK public procurement assistant. "
                "Use ONLY the retrieved contract evidence. "
                "Do not invent contracts, buyers, suppliers, values, dates, or categories. "
                "If a result is only partially relevant, say that clearly."
            )
        },
        {
            "role": "user",
            "content": f"""
User query:
{query}

Retrieved contracts:
{context}

Write the answer in this exact format:

Overall answer:
[Briefly answer the user query based only on the retrieved contracts.]

Most relevant contracts:
1. Notice ID: ...
   Title: ...
   Buyer: ...
   Suppliers: ...
   Why relevant: ...

2. Notice ID: ...
   Title: ...
   Buyer: ...
   Suppliers: ...
   Why relevant: ...

3. Notice ID: ...
   Title: ...
   Buyer: ...
   Suppliers: ...
   Why relevant: ...

Limitations:
[Clearly mention if any results are partial matches or if metadata is missing.]
"""
        }
    ]

    prompt = mistral_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(mistral_model.device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
            temperature=0.1,
            pad_token_id=mistral_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return mistral_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [31]:
def answer_query_with_mistral(query, top_k=5):
    retrieved_results = rag_search_v2(query, top_k=top_k)

    answer = generate_mistral_answer(
        query=query,
        results=retrieved_results
    )

    return answer, retrieved_results

### 11.1 Test the End-to-End RAG Pipeline

This runs retrieval, re-ranking, and Mistral answer generation for one assessment query.

In [32]:
query = "Find framework agreements for IT services in the NHS"

answer, retrieved_results = answer_query_with_mistral(query, top_k=5)

print(answer)
retrieved_results

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Overall answer:
Based on the retrieved contracts, here are the most relevant framework agreements for IT services in the NHS:

1. Notice ID: ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z
   Title: Extension- Digital Dictation, Speech/Voice Recognition, Outsourced Transcription and associated services framework agreement
   Buyer: NHS Commercial Solutions
   Suppliers: Aprobrium Ltd (trading as Lexacom), BigHand Limited, Crescendo Systems Ltd, DICT8 Limited, Dictate IT Limited, EpiQ Europe Ltd, Epiq Europe Ltd, G2 Speech UK Ltd, InventAsia Limited trading as Prescribe Digital, Nuance Communications Ireland Limited, OKS International Limited, ScribeTECH (UK) Ltd, T-Pro
   Why relevant: The contract is a framework agreement for IT services related to digital dictation, speech/voice recognition, outsourced transcription, and associated services, provided by multiple suppliers, and the buyer is NHS Commercial Solutions.

2. Notice ID: ocds-h6vhtk-02ba8e-2026-03-30T12:12:07+01:00
   Title: Digital

,Notice ID,Title,Buyer,Suppliers,Summary,Relevance Score,Hybrid Score,Why it matched
0,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...",NHS Commercial Solutions on behalf of Surrey a...,5.8426,0.0156,Semantic + BM25 candidate retrieval followed b...
1,ocds-h6vhtk-064cfe-2026-02-10T15:54:12Z,Consultancy and Advisory Services,NHS Shared Business Services Limited,None,This proposed Framework Agreement will aim to ...,5.7832,0.0164,Semantic + BM25 candidate retrieval followed b...
2,ocds-h6vhtk-02ba8e-2026-03-30T12:12:07+01:00,Digital Document Solutions (DDS) Framework Agr...,Guy's and St Thomas' NHS Foundation Trust,"3M MModal, ASL, Academia, Altodigital Networks...",The Digital Document Solutions (DDS) Framework...,5.6225,0.0230,Semantic + BM25 candidate retrieval followed b...
3,ocds-h6vhtk-058e51-2026-03-26T13:16:58Z,National Framework Agreement for the provision...,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,"Amvale Medical Transport Ltd, LSA Secure Ltd t...",The Countess of Chester Hospital NHS Foundatio...,5.3214,0.0258,Semantic + BM25 candidate retrieval followed b...
4,ocds-h6vhtk-0694e3-2026-05-08T15:04:53+01:00,Framework Agreement for the Provision of Leade...,NHS Wales Shared Services Partnership-Procurem...,None,Aneurin Bevan University Health Board (ABUHB) ...,5.2313,0.0251,Semantic + BM25 candidate retrieval followed b...


## 12. Manual Evaluation

Because no labelled benchmark exists for this procurement dataset, a manual relevance assessment is used.  
For each assessment query, the top-10 retrieved results are manually labelled:

- `2` = highly relevant  
- `1` = partially relevant  
- `0` = not relevant

In [33]:
assessment_eval_queries = [
    "Find framework agreements for IT services in the NHS",
    "Contracts awarded in London for construction",
    "Multi-supplier frameworks for cloud services"
]

In [34]:
manual_eval_rows = []

for query in assessment_eval_queries:
    results = rag_search_v2(query, top_k=10)

    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        manual_eval_rows.append({
            "query": query,
            "rank": rank,
            "notice_id": row["Notice ID"],
            "title": row["Title"],
            "buyer": row["Buyer"],
            "suppliers": row["Suppliers"],
            "summary": row["Summary"],
            "relevance_score": row["Relevance Score"],
            "hybrid_score": row["Hybrid Score"],
            "manual_relevance": ""  # Fill manually: 2, 1, or 0
        })

manual_eval_df = pd.DataFrame(manual_eval_rows)

manual_eval_df

,query,rank,notice_id,title,buyer,suppliers,summary,relevance_score,hybrid_score,manual_relevance
0,Find framework agreements for IT services in t...,1,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...",NHS Commercial Solutions on behalf of Surrey a...,5.8426,0.0156,
1,Find framework agreements for IT services in t...,2,ocds-h6vhtk-064cfe-2026-02-10T15:54:12Z,Consultancy and Advisory Services,NHS Shared Business Services Limited,None,This proposed Framework Agreement will aim to ...,5.7832,0.0164,
2,Find framework agreements for IT services in t...,3,ocds-h6vhtk-02ba8e-2026-03-30T12:12:07+01:00,Digital Document Solutions (DDS) Framework Agr...,Guy's and St Thomas' NHS Foundation Trust,"3M MModal, ASL, Academia, Altodigital Networks...",The Digital Document Solutions (DDS) Framework...,5.6225,0.0230,
3,Find framework agreements for IT services in t...,4,ocds-h6vhtk-058e51-2026-03-26T13:16:58Z,National Framework Agreement for the provision...,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,"Amvale Medical Transport Ltd, LSA Secure Ltd t...",The Countess of Chester Hospital NHS Foundatio...,5.3214,0.0258,
4,Find framework agreements for IT services in t...,5,ocds-h6vhtk-0694e3-2026-05-08T15:04:53+01:00,Framework Agreement for the Provision of Leade...,NHS Wales Shared Services Partnership-Procurem...,None,Aneurin Bevan University Health Board (ABUHB) ...,5.2313,0.0251,
5,Find framework agreements for IT services in t...,6,ocds-h6vhtk-05f100-2026-05-01T11:35:34+01:00,NHS LPP Asset Utilisation Framework Agreement,Guy's and St Thomas' NHS Foundation Trust,"Add Latent Ltd, Asset Workflows Limited, INNAX...",A framework opportunity for experienced and qu...,5.1559,0.0238,
6,Find framework agreements for IT services in t...,7,ocds-h6vhtk-067881-2026-03-31T09:11:38+01:00,Orthotic and Prosthetic Services (SBS10539),NHS Shared Business Services Limited,None,NHS Shared Business Services act in an Agency ...,5.1304,0.0318,
7,Find framework agreements for IT services in t...,8,ocds-h6vhtk-06001c-2026-03-25T09:36:08Z,Clinical Artificial Intelligence (CAI) Framework,Guy's and St Thomas' NHS Foundation Trust,None,This Pre-Market Engagement (PME) process is to...,5.0921,0.0247,
8,Find framework agreements for IT services in t...,9,ocds-h6vhtk-0693af-2026-05-07T15:31:52+01:00,D109 - Framework for Pharmaceutical Wholesale,University Hospitals of Derby and Burton NHS F...,None,This framework agreement is for the provision ...,5.0896,0.0081,
9,Find framework agreements for IT services in t...,10,ocds-h6vhtk-05f5e1-2026-03-16T18:15:37Z,SWL ICB Framework Agreement for the Provision ...,NHS SOUTH WEST LONDON INTEGRATED CARE BOARD,"ASHTONS HOSPITAL PHARMACY SERVICES LTD, AT Med...",NHS South West London Integrated Care Board (N...,4.9849,0.0098,


In [35]:
manual_eval_df.to_csv(
    "assessment_manual_evaluation_template.csv",
    index=False
)

### 12.1 Compute Evaluation Metrics

After manually filling `assessment_manual_evaluation_filled.csv`, the following metrics are computed:

- Precision@5 relaxed: labels 1 or 2 count as relevant  
- Precision@5 strict: only label 2 counts as relevant  
- MRR: reciprocal rank of the first highly relevant result  
- NDCG@5: ranking quality with graded relevance labels

In [37]:
manual_eval = pd.read_csv("assessment_manual_evaluation_filled.csv")
manual_eval["manual_relevance"] = manual_eval["manual_relevance"].astype(int)

manual_eval.head()

,query,rank,notice_id,title,buyer,suppliers,summary,relevance_score,hybrid_score,manual_relevance
0,Find framework agreements for IT services in t...,1,ocds-h6vhtk-0354e4-2026-01-06T14:46:47Z,"Extension- Digital Dictation, Speech/Voice Rec...",NHS Commercial Solutions,"Aprobrium Ltd (trading as Lexacom), BigHand Li...",NHS Commercial Solutions on behalf of Surrey a...,5.8426,0.0156,2
1,Find framework agreements for IT services in t...,2,ocds-h6vhtk-064cfe-2026-02-10T15:54:12Z,Consultancy and Advisory Services,NHS Shared Business Services Limited,NaN,This proposed Framework Agreement will aim to ...,5.7832,0.0164,0
2,Find framework agreements for IT services in t...,3,ocds-h6vhtk-02ba8e-2026-03-30T12:12:07+01:00,Digital Document Solutions (DDS) Framework Agr...,Guy's and St Thomas' NHS Foundation Trust,"3M MModal, ASL, Academia, Altodigital Networks...",The Digital Document Solutions (DDS) Framework...,5.6225,0.0230,2
3,Find framework agreements for IT services in t...,4,ocds-h6vhtk-058e51-2026-03-26T13:16:58Z,National Framework Agreement for the provision...,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,"Amvale Medical Transport Ltd, LSA Secure Ltd t...",The Countess of Chester Hospital NHS Foundatio...,5.3214,0.0258,0
4,Find framework agreements for IT services in t...,5,ocds-h6vhtk-0694e3-2026-05-08T15:04:53+01:00,Framework Agreement for the Provision of Leade...,NHS Wales Shared Services Partnership-Procurem...,NaN,Aneurin Bevan University Health Board (ABUHB) ...,5.2313,0.0251,0


In [38]:
#Relaxed (1 or 2 counts)
def precision_at_k(group, k=5, threshold=1):

    top_k = (
        group
        .sort_values("rank")
        .head(k)
    )

    return (
        top_k["manual_relevance"] >= threshold
    ).mean()

#Strict (only 2 counts)
def precision_at_k_strict(group, k=5):

    top_k = (
        group
        .sort_values("rank")
        .head(k)
    )

    return (
        top_k["manual_relevance"] == 2
    ).mean()

def mrr(group):

    ranked = (
        group
        .sort_values("rank")
    )

    for rank, rel in enumerate(
        ranked["manual_relevance"],
        start=1
    ):

        if rel == 2:
            return 1 / rank

    return 0


#NDCG@5
def dcg(relevances):

    relevances = np.array(relevances)

    return np.sum(
        (2**relevances - 1)
        /
        np.log2(
            np.arange(
                2,
                len(relevances) + 2
            )
        )
    )


def ndcg_at_5(group):

    labels = (
        group
        .sort_values("rank")
        .head(5)["manual_relevance"]
        .tolist()
    )

    ideal_labels = sorted(
        labels,
        reverse=True
    )

    ideal_dcg = dcg(
        ideal_labels
    )

    if ideal_dcg == 0:
        return 0

    return dcg(labels) / ideal_dcg

In [39]:
results = []

for query, group in manual_eval.groupby("query"):

    results.append({
        "query": query,
        "precision_at_5_relaxed":
            precision_at_k(
                group,
                k=5,
                threshold=1
            ),

        "precision_at_5_strict":
            precision_at_k_strict(
                group,
                k=5
            ),

        "mrr":
            mrr(group),

        "ndcg_at_5":
            ndcg_at_5(group)
    })

metrics_df = pd.DataFrame(results)

metrics_df

print("\nOverall metrics:")
print(metrics_df.mean(numeric_only=True))


Overall metrics:
precision_at_5_relaxed    0.666667
precision_at_5_strict     0.333333
mrr                       0.492063
ndcg_at_5                 0.880334
dtype: float64


In [40]:
print("Per Query Evaluation Results")
display(metrics_df)

Per Query Evaluation Results


,query,precision_at_5_relaxed,precision_at_5_strict,mrr,ndcg_at_5
0,Contracts awarded in London for construction,1.0,0.6,0.333333,0.774379
1,Find framework agreements for IT services in t...,0.4,0.4,1.000000,0.919721
2,Multi-supplier frameworks for cloud services,0.6,0.0,0.142857,0.946902


# 5. Evaluation

## Retrieval Quality Evaluation

As no publicly available labelled benchmark exists for the UK Find a Tender Service (FTS) procurement dataset, retrieval quality was evaluated using a manually curated relevance assessment set.

Three representative procurement queries were selected directly from the assessment brief:

* Find framework agreements for IT services in the NHS
* Contracts awarded in London for construction
* Multi-supplier frameworks for cloud services

For each query, the top retrieved contracts were manually reviewed and assigned one of the following relevance labels:

* **2** = Highly Relevant
* **1** = Partially Relevant
* **0** = Not Relevant

These manual labels were then used to calculate retrieval effectiveness metrics.

---

## Ranking Strategy

The retrieval pipeline uses a multi-stage ranking architecture designed to combine the strengths of both lexical and semantic retrieval.

### Stage 1: Intent Parsing

The user query is analysed to identify structured procurement concepts such as:

* NHS
* Framework Agreements
* Construction
* Cloud Services
* Geographic regions

These signals are used to support retrieval and filtering.

### Stage 2: Hybrid Retrieval

Two retrieval approaches are executed independently:

#### BM25 Retrieval

BM25 is used to identify contracts containing strong keyword matches. This is particularly useful for:

* Supplier names
* Framework identifiers
* CPV terminology
* Procurement-specific vocabulary

#### Dense Vector Retrieval

Semantic retrieval is performed using the **BAAI/bge-small-en-v1.5** embedding model stored within ChromaDB.

This enables concept-based matching even when exact keywords are not present.

### Stage 3: Reciprocal Rank Fusion (RRF)

Results from BM25 and vector retrieval are combined using Reciprocal Rank Fusion (RRF).

RRF was selected because it improves ranking robustness without requiring labelled training data.

### Stage 4: Cross-Encoder Re-ranking

The fused candidate set is re-ranked using a cross-encoder model.

This final stage performs query-document relevance scoring and improves the precision of the top-ranked results.

---

## Evaluation Metrics

The following retrieval metrics were calculated:

### Precision@5 (Relaxed)

Measures the proportion of retrieved contracts that were either partially or highly relevant.

### Precision@5 (Strict)

Measures the proportion of retrieved contracts that were highly relevant only.

### Mean Reciprocal Rank (MRR)

Measures how early the first highly relevant contract appears within the ranking.

### NDCG@5

Normalised Discounted Cumulative Gain evaluates ranking quality while considering different relevance levels and ranking positions.

---

## Evaluation Results

### Per Query Results

| Query                                                | Precision@5 (Relaxed) | Precision@5 (Strict) | MRR   | NDCG@5 |
| ---------------------------------------------------- | --------------------- | -------------------- | ----- | ------ |
| Contracts awarded in London for construction         | 1.000                 | 0.600                | 0.333 | 0.774  |
| Find framework agreements for IT services in the NHS | 0.400                 | 0.400                | 1.000 | 0.918  |
| Multi-supplier frameworks for cloud services         | 0.600                 | 0.000                | 0.143 | 0.947  |

### Overall Results

| Metric                | Score |
| --------------------- | ----- |
| Precision@5 (Relaxed) | 0.667 |
| Precision@5 (Strict)  | 0.333 |
| MRR                   | 0.492 |
| NDCG@5                | 0.880 |

The results indicate that the hybrid retrieval architecture is generally effective at ranking relevant procurement notices near the top of the result set. The strongest performance was observed for construction and NHS-related procurement searches.

---

## Failure Cases

Several failure modes were identified during evaluation:

### Terminology Mismatch

Cloud-related procurement notices are frequently described using alternative terminology such as:

* Digital Services
* Platform Services
* Hosting Services
* Infrastructure Services
* Software Solutions

As a result, some genuinely relevant contracts may not explicitly contain the phrase "Cloud Services", reducing retrieval precision.

### Incomplete Metadata

Some procurement notices contain missing:

* CPV classifications
* Supplier information
* Regional information

This limits the effectiveness of metadata-aware retrieval.

### Sparse Procurement Descriptions

Certain notices provide very limited descriptive text, making semantic matching more difficult.

---

## Optional Evaluation Approaches

### Manual Evaluation Set

A manually labelled relevance dataset was created for this assessment and used as the primary evaluation mechanism.

### Precision@k

Precision@5 was calculated using both relaxed and strict relevance criteria.

### LLM-as-Judge

LLM-as-Judge evaluation was considered but not implemented. Human relevance assessment was preferred because it provides a more transparent and trustworthy evaluation methodology for procurement retrieval tasks.


## Summary

The final system demonstrates:

- OCDS JSON ingestion and normalisation
- Contract-level retrieval schema
- Hugging Face BGE embeddings
- ChromaDB vector search
- BM25 keyword retrieval
- Reciprocal Rank Fusion
- Cross-encoder re-ranking
- Hugging Face Mistral answer generation
- Manual relevance evaluation with Precision@5, MRR, and NDCG@5